# Mix the real trays into the synthetic dataset

Produces one COCO dataset whose train/valid/test splits each hold both synthetic
trays and real 3DTRAB photographs, under the same 22-class taxonomy.

## The two split rules are different, and both matter

**Synthetic** is already **specimen-disjoint** — the generator assigns each tray a
split based on which specimens it drew from, so no individual insect appears on
both sides. Those assignments are carried in the filenames and are reused here
untouched.

**Real** must be **group-disjoint**. 3DTRAB is 19 tray/session groups, not 91
independent images: `3D01` alone is 31 frames of one physical tray, holding 1,449
of the 3,072 annotated insects (47%). Splitting real images individually would put
the same beetles on both sides of the evaluation, and the synthetic half's rigour
would not save you.

The `Sample_NNN_{40,42,44}cm` files are the same tray shot from three camera
heights, so they group too.

## Scale

At full-resolution export the two sources sit 1.14× apart in pixels per insect
(synthetic 129 px, real 148 px), against 1.97× before. That is inside the natural
spread across classes, so both are left at native resolution by default.

**Runs wherever the exported synthetic COCO lives.** CPU only.


In [ ]:
import os
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path
import json, shutil, re
from collections import defaultdict
import numpy as np
import pandas as pd

# Runs on either machine: Gaia to build the mix locally, Ares to rebuild it there
# after transferring insect_dataset_coco_20000_mix and 3DTRAB. The symlinks it writes
# are absolute, so the mix must be built on the machine that will train from it.
BASE_DIR = next((p for p in (Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares")),
                             Path.cwd()) if p.is_dir()), None)
assert BASE_DIR is not None, "base dir not found - set MA_BASE, e.g. export MA_BASE=/path/to/data"
print(f"machine: {BASE_DIR}")
SYN_COCO = BASE_DIR / "insect_dataset_coco_20000_mix"     # from the generator export
OUT_COCO = BASE_DIR / "insect_dataset_mixed"              # written here
TRAB_ZIP = BASE_DIR / "3DTRAB_Detection_Annotated_v1.zip"
TRAB_DIR = BASE_DIR / "3DTRAB_Detection_Annotated_v1"

SPLIT_FRACS   = {"train": 0.60, "valid": 0.20, "test": 0.20}   # of real INSECTS
REAL_OVERSAMPLE = 18     # times each real TRAIN image is repeated. Only ~46 of the 91
                        # real photographs land in train, so against ~20,000 synthetic
                        # trays the real half is a fraction of a percent of the boxes
                        # and barely reaches the loss. The report cell measures the
                        # actual share and suggests a value. Repeats cost no disk.
LINK_FILES = True       # True symlinks the images instead of copying them — instant,
                        # and costs no extra disk. Set False if the result has to be
                        # moved to another machine with a tool that does not follow
                        # links (plain `scp -r`, `zip`); with True, copy it using
                        # `rsync -aL` or `tar -czhf`, which dereference.
RESIZE_REAL_TO = None   # LEAVE THIS None. The generator now renders trays at 10.85
                        # px/mm, the same as the real ones, so the two sides are at
                        # matched resolution already. The residual per-class size
                        # differences (Coleoptera 1.86x, Diptera 2.57x, while Hemiptera
                        # and Araneae match) are a property of the SOURCE data — pitfall
                        # traps catch larger individuals than a taxonomic reference
                        # collection holds — not a resolution error. Resizing the real
                        # trays would shrink the classes that already match AND degrade
                        # the images you evaluate on. If the offset matters it will
                        # surface as those classes lagging in per-class AP on the real
                        # test split; that is the place to act on it, not here.
SEED = 0

# 3DTRAB label names -> the taxonomy used by the generator
NAME_MAP = {"Coleoptera": "Coleoptera", "Orthoptera": "Orthoptera",
            "Diptera": "Diptera", "Aranea": "Araneae", "Heteroptera": "Hemiptera"}

assert SYN_COCO.is_dir(), (
    f"no exported synthetic COCO at {SYN_COCO}\n"
    "Run generate_synthetic_trays_DATASET_server.ipynb first — its DATA_ROOT must "
    "match the one above.")
import shutil as _sh
_free = _sh.disk_usage(SYN_COCO).free / 1e9
print(f"disk at {BASE_DIR}: {_free:,.0f} GB free"
      + ("   (LINK_FILES=True, so the synthetic half costs nothing extra)"
         if _free > 20 else "   <-- tight; set LINK_FILES=True"))
_splits = [d.name for d in sorted(SYN_COCO.iterdir()) if d.is_dir()]
assert set(_splits) >= {"train", "valid", "test"}, f"unexpected splits: {_splits}"

SYN = {}
for sp in ("train", "valid", "test"):
    f = SYN_COCO / sp / "_annotations.coco.json"
    assert f.exists(), f"missing {f}"
    SYN[sp] = json.load(open(f))
CATS = sorted(SYN["train"]["categories"], key=lambda c: c["id"])
NAMES = [c["name"] for c in CATS]
NAME_TO_ID = {c["name"]: c["id"] for c in CATS}

print(f"synthetic: {SYN_COCO}")
for sp in ("train", "valid", "test"):
    print(f"   {sp:6s} {len(SYN[sp]['images']):6d} trays  "
          f"{len(SYN[sp]['annotations']):8d} insects")
print(f"\n{len(NAMES)} classes: {NAMES}")

# every split must agree on the categories, or ids mean different things
for sp in ("valid", "test"):
    other = sorted(SYN[sp]["categories"], key=lambda c: c["id"])
    assert [(c["id"], c["name"]) for c in other] == [(c["id"], c["name"]) for c in CATS], \
        f"{sp} has a different category list from train"
_unmapped = sorted({v for v in NAME_MAP.values()} - set(NAMES))
assert not _unmapped, f"3DTRAB classes absent from the synthetic taxonomy: {_unmapped}"
print("\ncategory lists agree across splits, and every 3DTRAB class exists in them")


In [ ]:
# ============================================================
# 3DTRAB -> COCO, in the synthetic taxonomy
# ============================================================
# Every insect is annotated as a polygon ("Insect Mask") plus a rectangle ("Insect
# BBox"); the ORDER is a separate `choices` region linked by id. Boxes come from the
# polygon where there is one.
#
# Annotators did not only correct the model's pre-annotations — they added 205 regions
# by hand (`"origin": "manual"`), and some of those they drew as a rectangle with NO
# polygon. A polygon-only reader silently drops them. Measured on this file:
#
#     3072  polygon, labelled
#       28  rectangle-only, labelled          <- recovered here
#        2  polygon whose label sits on its sibling bbox   <- recovered here
#       35  polygon, no label anywhere        -> unusable, trains as background
#       32  rectangle-only, no label          -> unusable, trains as background
#
# so 3,102 insects, not 3,072. Recovery is unambiguous: no id carries two `choices`
# regions, and across 2,340 insects labelled on BOTH their polygon and their bbox the
# two never disagree.
#
# NOTE: final_comparison.ipynb reads polygons only, so its 3DTRAB numbers are computed
# against 3,072 of these 3,102 boxes. Left as it is on purpose — this notebook is the
# one being changed.
import zipfile, urllib.parse
from PIL import Image

if not TRAB_DIR.exists():
    assert TRAB_ZIP.exists(), (f"neither {TRAB_DIR} nor {TRAB_ZIP} is here — copy the "
                               f"3DTRAB archive to this machine")
    print(f"unzipping {TRAB_ZIP} ...")
    with zipfile.ZipFile(TRAB_ZIP) as z: z.extractall(BASE_DIR)
TRAB_IMGS   = TRAB_DIR / "images"
TRAB_LABELS = TRAB_DIR / "labels" / "labels.json"
assert TRAB_IMGS.is_dir() and TRAB_LABELS.exists(), f"unexpected layout under {TRAB_DIR}"

def _sibling(rid):
    """insect_poly_7 <-> insect_bbox_7 — the same insect, pre-annotated as both."""
    m = re.match(r"insect_(poly|bbox)_(\d+)$", rid)
    return f"insect_{'bbox' if m.group(1) == 'poly' else 'poly'}_{m.group(2)}" if m else None

raw = json.load(open(TRAB_LABELS))
real_imgs, real_anns = [], []
unmapped, degenerate = defaultdict(int), 0
unlabelled_kept, dropped_imgs = 0, []
from_poly = from_rect = from_sibling_label = 0
for t in raw:
    fn = urllib.parse.unquote(t["data"]["image"]).replace("\\", "/").split("/")[-1]
    res = (t.get("annotations") or [{}])[0].get("result", [])
    # the order label lives on its own `choices` region, linked by region id
    ch = {r["id"]: (r["value"].get("choices") or [None])[0]
          for r in res if r.get("type") == "choices"}
    poly_ids = {r["id"] for r in res if r.get("type") == "polygonlabels"}
    W = H = None
    boxes = []
    n_unlab = 0
    for r in res:
        ty = r.get("type")
        rid = r.get("id", "")
        if ty == "polygonlabels":
            W, H = r["original_width"], r["original_height"]
            pts = r["value"]["points"]                   # PERCENT of W,H
            xs = [p[0]/100.0*W for p in pts]; ys = [p[1]/100.0*H for p in pts]
            x1, y1 = min(xs), min(ys); x2, y2 = max(xs), max(ys)
            sib = _sibling(rid)
            lab = ch.get(rid)
            if lab is None and sib and ch.get(sib):      # label sits on the bbox twin
                lab = ch[sib]; from_sibling_label += 1
            elif lab is not None:
                from_poly += 1
        elif ty == "rectanglelabels":
            sib = _sibling(rid)
            if sib and sib in poly_ids:
                continue                                 # same insect as a polygon above
            lab = ch.get(rid)
            if lab is None:
                n_unlab += 1; continue                   # a marked insect with no order
            W, H = r["original_width"], r["original_height"]
            v = r["value"]
            x1 = v["x"]/100.0*W; y1 = v["y"]/100.0*H
            x2 = x1 + v["width"]/100.0*W; y2 = y1 + v["height"]/100.0*H
            from_rect += 1
        else:
            continue
        x1, y1 = max(0.0, x1), max(0.0, y1)
        x2, y2 = min(float(W), x2), min(float(H), y2)
        if x2 - x1 < 1 or y2 - y1 < 1:
            degenerate += 1; continue
        if lab is None:
            n_unlab += 1; continue                       # unusable for a class model
        mapped = NAME_MAP.get(lab)
        if mapped is None:
            unmapped[lab] += 1; continue
        boxes.append((NAME_TO_ID[mapped], x1, y1, x2, y2))
    if W is None or not boxes:
        # every insect in this frame is unlabelled — including it would teach the
        # detector that real insects are background, so the whole image goes
        if n_unlab: dropped_imgs.append((fn, n_unlab))
        continue
    unlabelled_kept += n_unlab
    assert (TRAB_IMGS / fn).exists(), f"annotated file not on disk: {fn}"
    with Image.open(TRAB_IMGS / fn) as im: fw, fh = im.size
    assert (fw, fh) == (W, H), f"{fn}: labels say {W}x{H}, file is {fw}x{fh}"
    real_imgs.append({"file_name": fn, "width": W, "height": H})
    real_anns.append(boxes)

n_box = sum(len(b) for b in real_anns)
print(f"{len(real_imgs)} usable real images, {n_box} labelled insects "
      f"(of {len(raw)} records in labels.json)")
print(f"   from polygons {from_poly}"
      + (f", + {from_sibling_label} whose label sat on the sibling bbox" if from_sibling_label else "")
      + (f", + {from_rect} drawn by hand as a rectangle with no polygon" if from_rect else "")
      + (f" | {degenerate} sub-pixel regions skipped" if degenerate else ""))
for fn, n in dropped_imgs:
    print(f"   DROPPED {fn} — all {n} of its insects have no order label")
_noise = unlabelled_kept / (n_box + unlabelled_kept)
print(f"   {unlabelled_kept} insects inside the kept images have no order label "
      f"({_noise:.2%}) and will train as background — ordinary label noise at this "
      f"level" + ("" if _noise < 0.01 else ", but high enough to be worth dropping "
                  "those frames as well"))
print("   " + (f"UNMAPPED labels: {dict(unmapped)}" if unmapped else "no unmapped labels"))
assert not unmapped, "some 3DTRAB labels have no home in the taxonomy — extend NAME_MAP"

_id2n = {c["id"]: c["name"] for c in CATS}
_dist = defaultdict(int)
for b in real_anns:
    for cid, *_ in b: _dist[_id2n[cid]] += 1
print("   " + ", ".join(f"{k} {v}" for k, v in sorted(_dist.items(), key=lambda kv: -kv[1])))

# sanity: a tray insect is a small fraction of the frame, never most of it
_fr = np.array([[(x2-x1)/im["width"], (y2-y1)/im["height"]]
                for im, b in zip(real_imgs, real_anns) for _, x1, y1, x2, y2 in b])
print(f"   median box {np.median(_fr[:,0])*100:.2f}% x {np.median(_fr[:,1])*100:.2f}% of the frame")
assert np.median(_fr) < 0.25, "3DTRAB boxes look wrong — STOP"


## Split the real images by tray group

Groups are assigned to splits by hill climbing on a cost that balances both the
insect count and the per-class counts, so no split ends up without a class it is
meant to be evaluated on. Whole groups move together — that is the point.


In [ ]:
def group_of(fn):
    """Tray/session id. Frames sharing one are the SAME physical tray."""
    n = Path(fn).name
    m = re.match(r"Overview_(\d*D\d+)_", n)
    if m: return m.group(1)
    m = re.match(r"Sample_(\d+)_", n)
    if m: return "Sample_" + m.group(1)
    return "UNGROUPED_" + n

groups = defaultdict(list)
for k, im in enumerate(real_imgs):
    groups[group_of(im["file_name"])].append(k)
assert not any(g.startswith("UNGROUPED_") for g in groups), \
    f"unrecognised filenames: {[g for g in groups if g.startswith('UNGROUPED_')][:3]}"

CLS = sorted({NAMES[[c['id'] for c in CATS].index(cid)]
              for b in real_anns for cid, *_ in b})
G = np.array([[sum(1 for i in groups[g] for cid, *_ in real_anns[i]
                   if NAMES[[c['id'] for c in CATS].index(cid)] == cl) for cl in CLS]
              for g in sorted(groups)], float)
GN = sorted(groups)
TOT = G.sum(0)
SP = list(SPLIT_FRACS)
target = {s: TOT * SPLIT_FRACS[s] for s in SP}

# A class held by at least one group per split CAN reach every split, and should — a
# split without it cannot report AP for it at all, which is worth more than a few
# percent of balance. A class held by fewer groups than there are splits cannot be
# spread without breaking group disjointness, so it is exempt.
N_GROUPS_WITH = (G > 0).sum(0)
CAN_SPREAD = (N_GROUPS_WITH >= len(SP)) & (TOT >= 3 * len(SP))

def cost(asg):
    c = 0.0
    for j, s in enumerate(SP):
        got = G[asg == j].sum(0) if (asg == j).any() else np.zeros(len(CLS))
        c += (((got - target[s]) / np.maximum(target[s], 1.0)) ** 2).sum()
        c += 3.0 * ((got.sum() - target[s].sum()) / max(target[s].sum(), 1.0)) ** 2
        c += 25.0 * float(((got == 0) & CAN_SPREAD).sum())
    return c

rng = np.random.default_rng(SEED)
best = None
for _ in range(60):
    asg = rng.integers(0, len(SP), len(GN)); c = cost(asg); moved = True
    while moved:
        moved = False
        for i in range(len(GN)):
            cur = asg[i]
            for j in range(len(SP)):
                if j == cur: continue
                asg[i] = j; c2 = cost(asg)
                if c2 < c - 1e-12: c, cur, moved = c2, j, True
                else: asg[i] = cur
    if best is None or c < best[0]: best = (c, asg.copy())
_, ASG = best
REAL_SPLIT = {s: [i for k, g in enumerate(GN) if SP[ASG[k]] == s for i in groups[g]]
              for s in SP}

print(f"{len(groups)} groups, {len(real_imgs)} images, {n_box} insects\n")
hdr = f"{'split':7s}{'groups':>8s}{'images':>8s}{'insects':>9s}   " + \
      "".join(f"{c[:9]:>10s}" for c in CLS)
print(hdr)
for s in SP:
    idx = REAL_SPLIT[s]
    per = [sum(1 for i in idx for cid, *_ in real_anns[i]
               if NAMES[[c['id'] for c in CATS].index(cid)] == cl) for cl in CLS]
    ng = sum(1 for k, g in enumerate(GN) if SP[ASG[k]] == s)
    print(f"{s:7s}{ng:>8d}{len(idx):>8d}{sum(per):>9d}   " + "".join(f"{v:>10d}" for v in per))
print(f"{'target':7s}{'':8s}{'':8s}{n_box:>9d}   " +
      "".join(f"{int(t):>10d}" for t in TOT))

# leakage: a group must live in exactly one split
_seen = defaultdict(set)
for s in SP:
    for i in REAL_SPLIT[s]: _seen[group_of(real_imgs[i]["file_name"])].add(s)
_bad = {g: v for g, v in _seen.items() if len(v) > 1}
assert not _bad, f"groups spanning splits: {_bad}"
assert sum(len(v) for v in REAL_SPLIT.values()) == len(real_imgs)
print("\nno tray group appears in more than one split; every image placed exactly once")
_empty = [(s, cl) for s in SP for k, cl in enumerate(CLS)
          if sum(1 for i in REAL_SPLIT[s] for cid, *_ in real_anns[i]
                 if NAMES[[c['id'] for c in CATS].index(cid)] == cl) == 0]
for s, cl in _empty:
    k = CLS.index(cl)
    print(f"note: no real {cl} in {s} — it is held by only {int(N_GROUPS_WITH[k])} tray "
          f"group(s), so it cannot reach all {len(SP)} splits without leaking. "
          f"AP for {cl} on the {s} split comes from the synthetic half alone.")
_unavoidable = {cl for s, cl in _empty if not CAN_SPREAD[CLS.index(cl)]}
assert {cl for _, cl in _empty} == _unavoidable, \
    f"a class that COULD have been spread was left out of a split: " \
    f"{sorted({cl for _, cl in _empty} - _unavoidable)}"


## Merge

Synthetic images and annotations are copied through unchanged. Real ones are
appended with offset ids and a `real_` filename prefix, so nothing can collide
with a synthetic tray.


In [ ]:
# real ids start above every synthetic id, so a collision is arithmetically impossible
ID_OFFSET = 1 + max(max([i["id"] for i in SYN[s]["images"]] +
                        [a["id"] for a in SYN[s]["annotations"]] + [0])
                    for s in ("train", "valid", "test"))
print(f"real ids start at {ID_OFFSET:,} (highest synthetic id is {ID_OFFSET-1:,})\n")

def place(src, dst_path):
    """Put src at dst_path, as a link or a copy depending on LINK_FILES."""
    if LINK_FILES:
        try:
            dst_path.symlink_to(Path(src).resolve()); return
        except OSError:
            pass                      # filesystem refuses links — fall through
    shutil.copy2(src, dst_path)

if OUT_COCO.exists():
    print(f"removing previous {OUT_COCO}")
    shutil.rmtree(OUT_COCO)

report = []
for sp in ("train", "valid", "test"):
    dst = OUT_COCO / sp; dst.mkdir(parents=True, exist_ok=True)
    coco = {"info": {"description": f"synthetic + real 3DTRAB, {sp}"},
            "categories": CATS, "images": [], "annotations": []}

    # ── synthetic, untouched ──
    for im in SYN[sp]["images"]:
        src = SYN_COCO / sp / im["file_name"]
        if not src.exists(): continue
        if not (dst / im["file_name"]).exists():
            place(src, dst / im["file_name"])
        coco["images"].append(dict(im))
    _keep = {im["id"] for im in coco["images"]}
    coco["annotations"] = [dict(a) for a in SYN[sp]["annotations"] if a["image_id"] in _keep]
    n_syn_i, n_syn_a = len(coco["images"]), len(coco["annotations"])

    # ── real, appended ──
    # Only valid/test are ever seen once; TRAIN may repeat each photograph
    # REAL_OVERSAMPLE times so the real half actually reaches the loss. Repeats are
    # links to the first copy — the same pixels, not duplicated on disk.
    reps = REAL_OVERSAMPLE if sp == "train" else 1
    iid = ID_OFFSET; aid = ID_OFFSET
    for rep in range(reps):
        for i in REAL_SPLIT[sp]:
            im = real_imgs[i]
            first = dst / f"real_{im['file_name']}"
            name  = f"real_{rep}_{im['file_name']}" if rep else first.name
            out   = dst / name
            sc = RESIZE_REAL_TO / im["width"] if RESIZE_REAL_TO else 1.0
            W, H = round(im["width"] * sc), round(im["height"] * sc)
            if not out.exists():
                if rep:                                   # a repeat: reuse rep 0
                    place(first, out)
                elif RESIZE_REAL_TO:
                    # written back into the source container (.png stays lossless PNG),
                    # so resampling is the only thing that touches the pixels
                    with Image.open(TRAB_IMGS / im["file_name"]) as pim:
                        pim.convert("RGB").resize((W, H), Image.LANCZOS).save(out)
                else:
                    place(TRAB_IMGS / im["file_name"], out)
            coco["images"].append({"id": iid, "file_name": name, "width": W, "height": H,
                                   "source": "real"})
            for cid, x0, y0, x1, y1 in real_anns[i]:
                coco["annotations"].append({
                    "id": aid, "image_id": iid, "category_id": cid,
                    "bbox": [x0*sc, y0*sc, (x1-x0)*sc, (y1-y0)*sc],
                    "area": float((x1-x0)*(y1-y0)*sc*sc), "iscrowd": 0})
                aid += 1
            iid += 1
    json.dump(coco, open(dst / "_annotations.coco.json", "w"))
    report.append(dict(split=sp, syn_img=n_syn_i, syn_ann=n_syn_a,
                       real_img=len(coco["images"])-n_syn_i,
                       real_ann=len(coco["annotations"])-n_syn_a,
                       images=len(coco["images"]), anns=len(coco["annotations"])))
    print(f"{sp:6s} synthetic {n_syn_i:6d} img / {n_syn_a:8d} ann   "
          f"real {report[-1]['real_img']:4d} img / {report[-1]['real_ann']:5d} ann")

pd.DataFrame(report).to_csv(OUT_COCO / "mix_summary.csv", index=False)
print(f"\nwritten to {OUT_COCO}")


## Verify the written dataset

Re-reads the three JSON files from disk and checks them without trusting anything
still in memory. Every check is an assertion — if this cell prints its last line,
the dataset is sound.


In [ ]:
# ============================================================
# VERIFY
# ============================================================
CHK = {sp: json.load(open(OUT_COCO / sp / "_annotations.coco.json"))
       for sp in ("train", "valid", "test")}
ok = []

# 1. ids unique within a split, and every annotation points at a real image
for sp, c in CHK.items():
    ids = [i["id"] for i in c["images"]]
    aids = [a["id"] for a in c["annotations"]]
    assert len(ids) == len(set(ids)),   f"{sp}: duplicate image ids"
    assert len(aids) == len(set(aids)), f"{sp}: duplicate annotation ids"
    dangling = {a["image_id"] for a in c["annotations"]} - set(ids)
    assert not dangling, f"{sp}: {len(dangling)} annotations point at missing images"
ok.append("image and annotation ids are unique, and no annotation is orphaned")

# 2. categories identical in all three, and every used id is declared
for sp, c in CHK.items():
    assert [(x["id"], x["name"]) for x in sorted(c["categories"], key=lambda x: x["id"])] \
           == [(x["id"], x["name"]) for x in CATS], f"{sp}: category list drifted"
    used = {a["category_id"] for a in c["annotations"]} - {x["id"] for x in CATS}
    assert not used, f"{sp}: undeclared category ids {used}"
ok.append(f"all three splits declare the same {len(CATS)} categories")

# 3. every file is on disk, and the recorded size is the true size
import random
for sp, c in CHK.items():
    miss = [i["file_name"] for i in c["images"] if not (OUT_COCO / sp / i["file_name"]).exists()]
    assert not miss, f"{sp}: {len(miss)} files missing, e.g. {miss[:3]}"
    sample = random.Random(0).sample(c["images"], min(30, len(c["images"])))
    for i in sample:
        with Image.open(OUT_COCO / sp / i["file_name"]) as im: w, h = im.size
        assert (w, h) == (i["width"], i["height"]), \
            f"{sp}/{i['file_name']}: json says {i['width']}x{i['height']}, file is {w}x{h}"
ok.append("every image file exists; sampled sizes match the recorded width/height")

# 4. boxes are inside their image and have real area
for sp, c in CHK.items():
    dim = {i["id"]: (i["width"], i["height"]) for i in c["images"]}
    bad = 0
    for a in c["annotations"]:
        x, y, w, h = a["bbox"]; W, H = dim[a["image_id"]]
        if w <= 0 or h <= 0 or x < -1 or y < -1 or x + w > W + 1 or y + h > H + 1: bad += 1
    assert bad == 0, f"{sp}: {bad} boxes are degenerate or out of bounds"
ok.append("every box has positive area and lies inside its image")

# 5. LEAKAGE — real: no tray group in two splits. Read back off disk, not from memory.
def real_group(fn):
    return group_of(re.sub(r"^real_(\d+_)?", "", fn))
seen = defaultdict(set)
for sp, c in CHK.items():
    for i in c["images"]:
        if i["file_name"].startswith("real_"): seen[real_group(i["file_name"])].add(sp)
span = {g: sorted(v) for g, v in seen.items() if len(v) > 1}
assert not span, f"REAL LEAKAGE — tray groups in more than one split: {span}"
ok.append(f"{len(seen)} real tray groups, each confined to exactly one split")

# 6. LEAKAGE — no image file, real or synthetic, appears in two splits
base = defaultdict(set)
for sp, c in CHK.items():
    for i in c["images"]: base[i["file_name"]].add(sp)
dup = {f: sorted(v) for f, v in base.items() if len(v) > 1}
assert not dup, f"the same file is in two splits: {list(dup)[:3]}"
ok.append("no image file appears in more than one split")

# 7. the synthetic side came through untouched
for sp in ("train", "valid", "test"):
    src_i = {i["file_name"] for i in SYN[sp]["images"]}
    got_i = {i["file_name"] for i in CHK[sp]["images"] if not i["file_name"].startswith("real_")}
    assert got_i <= src_i, f"{sp}: synthetic images appeared that were not in the source"
    src_by_id = {a["id"]: a for a in SYN[sp]["annotations"]}
    for a in CHK[sp]["annotations"]:
        if a["id"] < ID_OFFSET:
            o = src_by_id[a["id"]]
            assert o["bbox"] == a["bbox"] and o["category_id"] == a["category_id"], \
                f"{sp}: synthetic annotation {a['id']} was altered"
ok.append("synthetic boxes and labels are byte-identical to the source export")

print("VERIFIED")
for line in ok: print("  ✓ " + line)


## What the mixed dataset looks like


In [ ]:
def origin(fn):
    """clean synthetic tray | brown synthetic tray | real photograph"""
    if fn.startswith("real_"):       return "real"
    if fn.startswith("brown_tray"):  return "synthetic brown"
    return "synthetic clean"

SRC3 = ["synthetic clean", "synthetic brown", "real"]
rows = []
for sp, c in CHK.items():
    o = {i["id"]: origin(i["file_name"]) for i in c["images"]}
    for a in c["annotations"]:
        rows.append((sp, o[a["image_id"]],
                     NAMES[[x["id"] for x in CATS].index(a["category_id"])],
                     np.sqrt(a["bbox"][2] * a["bbox"][3])))
D = pd.DataFrame(rows, columns=["split", "source", "class", "px"])
D["is_real"] = D.source == "real"

# ── clean : brown : real, by image and by box ──
print("composition — synthetic clean : synthetic brown : real\n")
for unit in ("images", "insects"):
    print(f"  by {unit}")
    tab = []
    for sp, c in CHK.items():
        if unit == "images":
            n = {k: sum(1 for i in c["images"] if origin(i["file_name"]) == k) for k in SRC3}
        else:
            n = {k: int((D[(D.split == sp) & (D.source == k)]).shape[0]) for k in SRC3}
        tot = max(sum(n.values()), 1)
        tab.append({"split": sp, **n, "total": sum(n.values()),
                    "ratio": " : ".join(f"{100*n[k]/tot:.1f}%" for k in SRC3)})
    print("    " + pd.DataFrame(tab).to_string(index=False).replace("\n", "\n    "))
    print()

t = pd.DataFrame([{"split": sp,
                   "syn img": sum(1 for i in c["images"] if not i["file_name"].startswith("real_")),
                   "real img": sum(1 for i in c["images"] if i["file_name"].startswith("real_")),
                   "syn ann": int((D[(D.split == sp) & (~D.is_real)]).shape[0]),
                   "real ann": int((D[(D.split == sp) & (D.is_real)]).shape[0])}
                  for sp, c in CHK.items()])
t["real share of boxes"] = (t["real ann"] / (t["real ann"] + t["syn ann"])).map("{:.2%}".format)
print(t.to_string(index=False))

# How much of the training loss actually comes from real photographs?
_tr = t[t.split == "train"].iloc[0]
_share = _tr["real ann"] / (_tr["real ann"] + _tr["syn ann"])
print(f"\nreal boxes are {_share:.2%} of the training signal, at "
      f"REAL_OVERSAMPLE = {REAL_OVERSAMPLE}")
if _share < 0.02:
    _need = int(np.ceil(0.05 * _tr["syn ann"] / max(_tr["real ann"] / REAL_OVERSAMPLE, 1)))
    print(f"   At this level the real half barely reaches the loss and the run is, in\n"
          f"   effect, still synthetic-only. REAL_OVERSAMPLE = {_need} would put it near\n"
          f"   5%. The trade is influence against memorising "
          f"{int(_tr['real img'] / REAL_OVERSAMPLE)} photographs — repeats are the same\n"
          f"   pixels every epoch, so watch real-only validation AP, not the mixed number.")
elif _share > 0.25:
    print("   The real half now dominates. That is fine only if you meant to fine-tune\n"
          "   rather than pre-train.")

print("\n\ninsect size, sqrt(area) in pixels\n")

# PER CLASS, and only for the classes 3DTRAB actually annotates. The pooled medians
# are NOT comparable: the real side is 85% Coleoptera, while the synthetic side is
# balanced across 22 orders and full of small-bodied ones (Thysanoptera renders at
# ~25 px). Comparing those two medians measures the taxonomy mix, not the scale, and
# an earlier version of this cell used it to recommend RESIZE_REAL_TO — which would
# have shrunk the real trays to fix a difference that was never about size.
_shared = sorted(set(D[D.is_real]["class"]) & set(D[~D.is_real]["class"]))
print(f"  {'class':16s}{'real':>8s}{'n':>7s}{'synthetic':>11s}{'ratio':>8s}")
_off = []
for _c in _shared:
    _rv = D[(D.is_real) & (D["class"] == _c)].px
    _sv = D[(~D.is_real) & (D["class"] == _c)].px
    if not len(_rv) or not len(_sv):
        continue
    _rm, _sm = float(_rv.median()), float(_sv.median())
    _ratio = _rm / max(_sm, 1e-9)
    _tag = "matched" if 0.8 <= _ratio <= 1.25 else ""
    print(f"  {_c:16s}{_rm:>8.0f}{len(_rv):>7d}{_sm:>11.0f}{_ratio:>8.2f}x   {_tag}")
    if not (0.8 <= _ratio <= 1.25):
        _off.append((_c, _ratio, len(_rv)))

_rp, _sp = D[D.is_real].px.median(), D[~D.is_real].px.median()
print(f"\n  pooled: real {_rp:.0f} px vs synthetic {_sp:.0f} px ({_rp/_sp:.2f}x) — "
      f"NOT a like-for-like\n  comparison, since the two sides have different class "
      f"mixes. Use the per-class rows.")

if _off:
    _w = sum(n for _, _, n in _off) / max(len(D[D.is_real]), 1)
    print(f"\n  Off by more than 25%: "
          + ", ".join(f"{c} {r:.2f}x" for c, r, _ in _off)
          + f"  ({_w:.0%} of real boxes)")
    print("  This is a property of the SOURCE data, not a rendering error — a rendering\n"
          "  error would shift every class equally. Pitfall traps select for larger\n"
          "  individuals than a taxonomic reference collection contains. Do NOT set\n"
          "  RESIZE_REAL_TO to 'fix' it: that shrinks the classes that already match and\n"
          "  degrades the real images you evaluate on. If it matters, it will show up as\n"
          "  these classes specifically lagging in per-class AP on the real test split.")
else:
    print("\n  Every shared class is within 25% — synthetic and real are at the same scale.")

print("\n\nclasses that now have real examples\n")
p = (D[D.is_real].pivot_table(index="class", columns="split", values="px",
                                       aggfunc="count", fill_value=0)
     .reindex(columns=["train", "valid", "test"], fill_value=0))
p["total"] = p.sum(1)
p["synthetic"] = [int((D[(~D.is_real) & (D["class"] == c)]).shape[0]) for c in p.index]
print(p.sort_values("total", ascending=False).to_string())
print(f"\n{len(CATS) - len(p)} of {len(CATS)} classes are synthetic-only — expected, "
      f"3DTRAB only annotates {len(p)} orders.")


In [ ]:
# ============================================================
# One figure for the report / supervisor
# ============================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
w = 0.38; x = np.arange(3); sp3 = ["train", "valid", "test"]
ax[0].bar(x - w/2, [int(t[t.split == s]["syn img"].iloc[0]) for s in sp3], w,
          label="synthetic (clean + brown)", color="#4C78A8")
ax[0].bar(x + w/2, [int(t[t.split == s]["real img"].iloc[0]) for s in sp3], w,
          label="real 3DTRAB", color="#E45756")
ax[0].set_xticks(x); ax[0].set_xticklabels(sp3); ax[0].set_yscale("log")
ax[0].set_ylabel("images (log)"); ax[0].set_title("Images per split"); ax[0].legend(frameon=False)

for src, col in ((False, "#4C78A8"), (True, "#E45756")):
    d = D[D.is_real == src]; nm = "real 3DTRAB" if src else "synthetic"
    ax[1].hist(d.px, bins=60, range=(0, 400), density=True,
               alpha=.55, color=col, label=f"{nm} (median {d.px.median():.0f} px)")
ax[1].set_xlabel("insect size, sqrt(area) in px"); ax[1].set_ylabel("density")
ax[1].set_title("Scale match"); ax[1].legend(frameon=False)

cc = p.sort_values("total", ascending=False)
ax[2].barh(range(len(cc)), cc["total"], color="#E45756")
ax[2].set_yticks(range(len(cc))); ax[2].set_yticklabels(cc.index, fontsize=9)
ax[2].invert_yaxis(); ax[2].set_xlabel("real annotated insects")
ax[2].set_title("Real coverage by class")
fig.suptitle(f"{OUT_COCO.name} — synthetic trays mixed with real 3DTRAB photographs",
             fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig(OUT_COCO / "mix_overview.png", dpi=130, bbox_inches="tight")
plt.close(fig)
D.to_csv(OUT_COCO / "mix_boxes.csv", index=False)
pd.DataFrame([{"group": g, "split": list(v)[0], "images": len(groups[g]),
               "insects": sum(len(real_anns[i]) for i in groups[g])}
              for g, v in sorted(seen.items())]).to_csv(OUT_COCO / "real_split_groups.csv",
                                                        index=False)
print(f"mix_overview.png, mix_boxes.csv, real_split_groups.csv, mix_summary.csv "
      f"-> {OUT_COCO}")
print(f"\nTrain on {OUT_COCO} exactly as you trained on the synthetic-only export — "
      f"same layout, same class ids.")


In [ ]:
# ============================================================
# The held-out real images, on their own, for final_comparison.ipynb
# ============================================================
# Once a model trains on this mixed dataset, scoring it against the WHOLE of
# 3DTRAB is contaminated — half those photographs were in its training set. This
# writes the test-split real images as a standalone COCO file, in the same shape
# as the `3dtrab_coco.json` final_comparison already reads, so the real-data
# number in the comparison stays honest.
_te = CHK["test"]
_real = [i for i in _te["images"] if i["file_name"].startswith("real_")]
_rid = {i["id"] for i in _real}
_held = {"info": {"description": "3DTRAB, TEST-SPLIT ONLY — safe to score a model "
                                 "trained on insect_dataset_mixed"},
         "categories": CATS,
         "images": [{"id": i["id"],
                     "file_name": re.sub(r"^real_(\d+_)?", "", i["file_name"]),
                     "width": i["width"], "height": i["height"]} for i in _real],
         "annotations": [dict(a) for a in _te["annotations"] if a["image_id"] in _rid]}
json.dump(_held, open(OUT_COCO / "3dtrab_coco_TESTSPLIT.json", "w"))

_tr_files = sorted({re.sub(r"^real_(\d+_)?", "", i["file_name"])
                    for sp in ("train", "valid") for i in CHK[sp]["images"]
                    if i["file_name"].startswith("real_")})
(OUT_COCO / "3dtrab_seen_in_training.txt").write_text("\n".join(_tr_files) + "\n")

print(f"3dtrab_coco_TESTSPLIT.json   {len(_held['images'])} images, "
      f"{len(_held['annotations'])} insects — point final_comparison's TRAB_COCO here")
print(f"3dtrab_seen_in_training.txt  {len(_tr_files)} images the model will have seen; "
      f"never score on these")
print(f"\n   Both index the ORIGINAL filenames in {TRAB_DIR.name}/images/, so no image "
      f"needs copying.")


## What to do next

1. Copy `insect_dataset_mixed/` to the training machine. Its layout and class ids are
   identical to the synthetic-only export, so the RF-DETR and flatbug configs need
   only their dataset path changed.

   With `LINK_FILES = True` (the default) the images are symlinks, so copy with a
   tool that follows them — `rsync -aL src/ host:dst/` or `tar -czhf mixed.tgz
   insect_dataset_mixed`. Plain `scp -r` and `zip` copy the links themselves and
   you arrive with a folder of dead pointers. Set `LINK_FILES = False` to sidestep
   it entirely at the cost of the disk.
2. Check the **real share of the training signal** printed above. Below ~2% the run
   is synthetic-only in all but name; raise `REAL_OVERSAMPLE` and re-run this
   notebook (repeats are links, so it costs no disk and takes seconds).
3. When scoring on real data afterwards, point `final_comparison.ipynb` at
   `3dtrab_coco_TESTSPLIT.json`, **not** the full 3DTRAB set. Roughly half of
   3DTRAB is now training data, and a number computed over all 91 images would be
   measuring memorisation. `3dtrab_seen_in_training.txt` lists exactly which files
   are off limits.
4. The synthetic-only baseline is still the honest comparison point: same
   architecture, same schedule, `SYN_COCO` instead of `OUT_COCO`. The difference
   between the two on `3dtrab_coco_TESTSPLIT.json` is what mixing bought you.
